# 41. Parsing HTML — `beautifulsoup4 · lxml`
**Python Libraries Lab · Phase 3 · Web & Networking**

_Pull data out of web pages and HTML files._

✅ **Runs in Colab** as-is.

Web pages are HTML, text with tags like h two and div arranged in a tree. BeautifulSoup reads that tree and lets you search it, find every product card, read the price inside each, and follow links. lxml is a very fast parser that BeautifulSoup can use, and it also supports XPath, a precise query language for HTML and XML.

**In this notebook:** This notebook creates a small three page ShopEasy deals site, so it works without the internet. You will parse a page, extract every product into a table, follow the next page links, clean the prices into numbers, and do the same with lxml and XPath.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q beautifulsoup4 lxml
import importlib.metadata as md
for pkg in ['beautifulsoup4', 'lxml']:
    print(pkg, md.version(pkg))

## Step 1 · Make the practice website
Three HTML pages of products, written to a `site` folder.

In [ ]:
from pathlib import Path
site = Path("site"); site.mkdir(exist_ok=True)
products = [("Steel Bottle", 499, 4.5, "Kitchen"), ("Yoga Mat", 1299, 4.1, "Fitness"), ("Desk Lamp", 899, 3.9, "Home"),
            ("Notebook", 149, 4.8, "Stationery"), ("Headphones", 2499, 4.3, "Electronics"), ("Water Jug", 349, 3.5, "Kitchen"),
            ("Dumbbells", 1999, 4.6, "Fitness"), ("Table Fan", 1799, 4.0, "Home"), ("Gel Pens", 99, 4.2, "Stationery")]
pages = [products[0:3], products[3:6], products[6:9]]
for i, page in enumerate(pages, 1):
    cards = "\n".join(f'''<div class="product" data-category="{c}">
  <h2 class="name">{n}</h2><span class="price">Rs. {p:,}</span>
  <span class="rating" title="{r} out of 5">{r}</span><a href="/item/{n.lower().replace(' ', '-')}">details</a>
</div>''' for n, p, r, c in page)
    nxt = f'<a class="next" href="page{i + 1}.html">Next page</a>' if i < len(pages) else ""
    (site / f"page{i}.html").write_text(f"<html><head><title>ShopEasy page {i}</title></head><body><h1>ShopEasy deals</h1>{cards}{nxt}</body></html>")
print("Wrote", [p.name for p in sorted(site.glob("*.html"))])

## Step 2 · Parse a page
`BeautifulSoup(html, 'lxml')` builds the tree. Then navigate it.

In [ ]:
from bs4 import BeautifulSoup
html = (site / "page1.html").read_text()
soup = BeautifulSoup(html, "lxml")
print("Title:", soup.title.string)
print("Heading:", soup.h1.get_text())
first = soup.find("div", class_="product")
print(first.prettify()[:300])

## Step 3 · Extract every product
`select` uses CSS selectors. Read text, attributes and clean the price into a number.

In [ ]:
def parse_products(soup):
    rows = []
    for card in soup.select("div.product"):
        rows.append({
            "name": card.select_one(".name").get_text(strip=True),
            "price": int(card.select_one(".price").get_text().replace("Rs.", "").replace(",", "").strip()),
            "rating": float(card.select_one(".rating").get_text()),
            "category": card["data-category"],
            "link": card.a["href"],
        })
    return rows
for row in parse_products(soup):
    print(row)

## Step 4 · Follow the next-page links
Keep going while there is a 'Next page' link. Then load everything into pandas.

In [ ]:
import pandas as pd
all_rows, page = [], "page1.html"
while page:
    s = BeautifulSoup((site / page).read_text(), "lxml")
    all_rows += parse_products(s)
    nxt = s.select_one("a.next")
    page = nxt["href"] if nxt else None
df = pd.DataFrame(all_rows)
print(len(df), "products from all pages")
df.sort_values("rating", ascending=False).head()

## Step 5 · lxml with XPath
XPath is another way to point at parts of a page; very fast with lxml.

In [ ]:
from lxml import html as lhtml
tree = lhtml.fromstring((site / "page2.html").read_text())
names = tree.xpath('//div[@class="product"]/h2/text()')
kitchen = tree.xpath('//div[@data-category="Kitchen"]/h2/text()')
prices = [p.replace("Rs. ", "") for p in tree.xpath('//span[@class="price"]/text()')]
print(names, kitchen, prices)

## Step 6 · Optional: a real practice site
books.toscrape.com is a site made for scraping practice. Skips politely without internet.

In [ ]:
import requests
try:
    r = requests.get("https://books.toscrape.com/", timeout=10, headers={"User-Agent": "shopeasy-lab/1.0"})
    books = BeautifulSoup(r.text, "lxml").select("article.product_pod")
    for b in books[:5]:
        print(b.h3.a["title"][:45], "|", b.select_one(".price_color").get_text())
except requests.RequestException as e:
    print("Could not reach the practice site right now:", type(e).__name__)

## ✍️ Your turn
Using `df`, store the **name of the cheapest product** across all pages in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == "Gel Pens", f"Expected 'Gel Pens', got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = df.loc[df["price"].idxmin(), "name"]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **HTML tree**
- **Tag / attribute**
- **find / find_all**
- **CSS selector**
- **XPath**
- **Scraping ethics**

**Next:** 42. Scrapy Crawlers (`scrapy`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/42_scrapy.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install beautifulsoup4 lxml
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.